# Stroke Prevention Demo Final Baseline Model (Logistic Regression)

DP1 Contributor: Erik Bergmark (erikcb2)  
DP2 Contributor: Joshua Lee (jcl12)  
Consolidated by: Daryl Okeke (dokek2)

## Purpose
This notebook trains and evaluates a baseline model to predict `stroke` using a clean workflow:
- Drops known leakage columns
- Uses fixed 5-fold stratified cross-validation for evaluation
- Trains logistic regression with class imbalance handling
- Evaluates using out-of-fold predictions across all rows
- Produces a risk score (probability) plus a simple demo threshold for yes or no

## How to run
Run cells from top to bottom.

## What most people will tweak
Go to Section 1 Config. Most changes should be made there.


## 1. Config

This section is the main place to tweak the baseline without touching the rest.

Key ideas:
- `CV_FOLDS` and `RANDOM_STATE` control evaluation reproducibility.
- `LEAKAGE_COLS` are columns we remove because they can act like post stroke proxy signals.
- `NOMINAL_CATEGORICAL_COLS` and `ORDINAL_COLS` define the categorical pipelines.
- `THRESHOLD` controls how strict we are when turning a probability into a yes or no prediction for metrics.


In [20]:
from pathlib import Path

# Cross-validation settings (keep fixed for reproducibility)
CV_FOLDS = 5
RANDOM_STATE = 42
CV_SHUFFLE = True

# Demo threshold for turning probability into 0 or 1 predictions
# Lower threshold catches more stroke cases but creates more false alarms
THRESHOLD = 0.30

# Data location (relative to repo root)
DATA_REL_PATH = Path("data/raw/stroke_data.csv")

# Output report location (relative to repo root)
REPORT_REL_PATH = Path("reports/baseline_metrics.md")

# Confirmed leakage columns — dropped before the split so they never reach the model
# Coronary Heart Disease added: cross-sectional data means CHD can be diagnosed *after*
# a stroke during follow-up care, giving a 3x stroke rate that inflates performance
LEAKAGE_COLS = [
    "General health condition",
    "depression",
    "Minutes sedentary activity",
    "Coronary Heart Disease",
]

# Columns excluded for practical/UX reasons — require lab tests most users won't have
# HDL, LDL, and Triglycerides need a lipid panel; dropping them keeps the app accessible
# Note: these also carry treatment confounding (post-stroke statin use lowers LDL/raises HDL)
EXCLUDE_COLS = [
    "High-density lipoprotein",
    "Triglyceride",
    "Low-density lipoprotein",
]

# Columns that are nearly redundant with other features and add multicollinearity
# Total fat r=0.999 with sum of the 3 fatty acid columns — keeping the components is more informative
REDUNDANT_COLS = [
    "Total fat",
]

# Nominal categorical columns (no meaningful order between values)
# One-hot encoded — includes BMI because the stroke rate is non-monotonic across BMI groups
NOMINAL_CATEGORICAL_COLS = [
    "gender",
    "Race",
    "Marital status",
    "sleep disorder",
    "Health Insurance",
    "Body Mass Index",
]

# Ordinal categorical columns (meaningful rank order)
# Each list inside ORDINAL_CATEGORIES is the ordered levels for the matching column
# age: stroke rate goes 1.9% -> 6.1% -> 12.3% — strong monotonic signal
ORDINAL_COLS = ["age"]
ORDINAL_CATEGORIES = [[1, 2, 3]]

# Note: binary 0/1 columns (alcohol, smoke, diabetes, hypertension, high cholesterol)
# are passed through the numeric pipeline as-is — already 0/1 scale, one-hot would
# just create perfectly collinear dummy pairs with no information gain.
#
# Note on maybe-leakage: hypertension, diabetes, high cholesterol, smoke, alcohol,
# sleep disorder, and sleep time are flagged as potential post-stroke proxies in
# docs/reference/leakage_candidates.md. We keep them deliberately for this baseline.

# Dietary/nutritional columns where a recorded value of 0 is physiologically implausible
# A full-day dietary recall of zero calories, protein, fat, etc. indicates a missing response
# These zeros are replaced with NaN before the split so the imputer handles them properly
ZERO_AS_MISSING_COLS = [
    "energy",
    "protein",
    "Carbohydrate",
    "Dietary fiber",
    "Total saturated fatty acids",
    "Total monounsaturated fatty acids",
    "Total polyunsaturated fatty acids",
    "Potassium",
    "Sodium",
]

# Hard physiological caps — applied in Section 3b before any model fitting
# These enforce outer limits of what is biologically achievable, not typical ranges.
# Values outside these bounds are almost certainly data entry errors or coding artifacts.
# Caps are set generously above the p99 of the observed distribution so we do not
# clip genuine extreme-but-real values — we only remove the truly impossible ones.
#
# Dietary recall note (from data dictionary): NHANES 24-hour dietary recall (DR1T)
# is a single-day self-report. Single-day recall is well-documented to produce extreme
# outliers from respondents mis-estimating portion sizes. The data dictionary also notes
# NHANES uses special missing codes (7-fill, 9-fill) which were pre-cleaned here, but
# residual entry errors in continuous fields are common.
#
# Format: column_name -> (min_allowed, max_allowed). None = no cap in that direction.
PHYSIOLOGICAL_CAPS = {
    # Dietary recall (24-hour; NHANES DR1T variables)
    # Basis: outer edge of human physiological capacity even for extreme diets.
    # p99 values from this dataset shown in comments for reference.
    "energy":                             (400,   6000),  # kcal/day  | p99 ≈ 5049  | max=13687
    "protein":                            (5,     280),   # g/day     | p99 ≈ 214   | max=387
    "Carbohydrate":                       (5,     700),   # g/day     | p99 ≈ 622   | max=1815
    "Dietary fiber":                      (1,     70),    # g/day     | p99 ≈ 51    | max=107
    "Total saturated fatty acids":        (0.5,   100),   # g/day     | p99 ≈ 77    | max=206
    "Total monounsaturated fatty acids":  (0.5,   100),   # g/day     | p99 ≈ 83    | max=222
    "Total polyunsaturated fatty acids":  (0.5,   75),    # g/day     | p99 ≈ 58    | max=147
    "Potassium":                          (200,   8000),  # mg/day    | p99 ≈ 6473  | max=14812
    "Sodium":                             (300,   12000), # mg/day    | p99 ≈ 9157  | min=7 (impossible)
    # Clinical
    # Glycohemoglobin (HbA1c) below 3.5% is below the analytical detection range
    # of clinical assays — observed min of 2.0 is a measurement/coding artifact
    "Glycohemoglobin":                    (3.5,   18.0),  # %         | p1  ≈ 4.6   | min=2.0
}

# Outlier clipping settings (IQR-based, applied per CV fold)
# Bounds are fit on each training fold only.
CLIP_OUTLIERS = True
CLIP_IQR_MULTIPLIER = 2.0

# Elastic net logistic regression settings
# Elastic net combines L1 (lasso) and L2 (ridge) penalties:
#   - L1 drives irrelevant feature weights to exactly zero (feature selection)
#   - L2 shrinks correlated features together rather than picking one arbitrarily
#   - l1_ratio blends them: 0.0 = pure ridge, 1.0 = pure lasso, 0.5 = equal mix
# solver="saga" is required for elasticnet penalty; it handles large sparse problems well
# C is the inverse regularization strength: smaller C = more regularization
CLASS_WEIGHT = "balanced"
MAX_ITER     = 2000
SOLVER       = "saga"
PENALTY      = "elasticnet"
L1_RATIO     = 0.5   # equal mix of L1 and L2
C            = 1   # tuned via 5-fold CV grid search; tighter regularization prevents
               # the model from fitting noise in weak dietary features (C=0.1 vs 1.0: +0.003 CV AUC)

# Thresholds we quickly scan for a precision/recall tradeoff check
THRESHOLD_SCAN = [0.05, 0.10, 0.20, 0.30]


## 2. Imports

In [21]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler


## 3. Load data

This notebook expects the CSV at `data/raw/stroke_data.csv` in the repo.

We find the repo root by searching upward for that file.


In [22]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / DATA_REL_PATH).exists():
            return p
    return here

ROOT = find_repo_root()
DATA_PATH = ROOT / DATA_REL_PATH

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head()


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


## 3b. Data cleaning

Steps applied before splitting (order matters):

1. **Strip column names** — fixes trailing whitespace (`alcohol ` → `alcohol`)
2. **Duplicate row check** — report and drop any exact duplicates
3. **Zero → NaN for dietary columns** — zero daily intake of calories, fat, protein, etc. is physiologically impossible and signals a missing dietary recall response
4. **Physiological hard caps** — clamp values that exceed the outer limits of what the human body can produce or consume; these are data entry artifacts, not real outliers
5. **Data quality summary** — print min/max/missing after cleaning so problems are visible

### Why hard caps and not just IQR clipping?

IQR clipping is statistical — it finds outliers relative to the distribution of the data. For **highly skewed dietary recall columns**, the IQR lower bound can go negative, meaning a Sodium value of 7 mg/day (physiologically impossible — a single sip of most beverages contains more than that) would survive IQR clipping entirely.

Hard caps use domain knowledge instead: they enforce the outer edge of what is biologically achievable. We set them **generously above the p99** of the observed distribution so we do not trim genuine extreme-but-real values — we only remove entries that could not exist in a living person.

The data dictionary notes that NHANES 24-hour dietary recall (DR1T) is a single-day self-report, which is well-documented to produce extreme outliers from respondents mis-estimating portion sizes or reporting multi-day intake for a single day. These values have no signal and should not influence the model.

In [23]:
# --- Strip whitespace from column names ---
original_cols = df.columns.tolist()
df.columns = df.columns.str.strip()
changed = [(o, n) for o, n in zip(original_cols, df.columns.tolist()) if o != n]
if changed:
    print("Column names stripped (trailing/leading whitespace removed):")
    for old, new in changed:
        print(f"  '{old}'  ->  '{new}'")
else:
    print("No column name whitespace issues found.")

# --- Duplicate row check ---
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows found: {n_dupes}")
if n_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dropped {n_dupes} duplicate rows. Dataset now has {len(df)} rows.")

Column names stripped (trailing/leading whitespace removed):
  'alcohol '  ->  'alcohol'

Duplicate rows found: 0


In [24]:
# --- Replace 0 with NaN in dietary columns ---
# A full-day dietary recall reporting zero calories, fat, protein, etc. means
# the participant did not complete that section, not that they consumed nothing.
# Replacing with NaN lets the median imputer fill these in during preprocessing.
zero_cols_present = [c for c in ZERO_AS_MISSING_COLS if c in df.columns]
zero_counts = (df[zero_cols_present] == 0).sum()

df[zero_cols_present] = df[zero_cols_present].replace(0, np.nan)

print("Zeros replaced with NaN in dietary columns:")
print(f"{'Column':<45} {'Values replaced':>15}")
print("-" * 62)
any_replaced = False
for col in zero_cols_present:
    n = zero_counts[col]
    if n > 0:
        print(f"{col:<45} {n:>15}")
        any_replaced = True
if not any_replaced:
    print("  None found.")
print(f"\nTotal values replaced: {zero_counts.sum()}")

Zeros replaced with NaN in dietary columns:
Column                                        Values replaced
--------------------------------------------------------------
energy                                                      1
protein                                                     2
Carbohydrate                                                1
Dietary fiber                                               5
Total saturated fatty acids                                 2
Total monounsaturated fatty acids                           2
Total polyunsaturated fatty acids                           2
Potassium                                                   1

Total values replaced: 16


In [25]:
# --- Apply physiological hard caps ---
# Clamp values that exceed what is biologically achievable.
# Caps are defined in PHYSIOLOGICAL_CAPS in Section 1 Config.
# Applied before CV splitting — these are domain-knowledge rules,
# not statistics, so there is no leakage risk in applying them to the full dataset.

cap_cols_present = [c for c in PHYSIOLOGICAL_CAPS if c in df.columns]
total_capped = 0

print("Physiological hard caps applied:")
print(f"{'Column':<42} {'Min cap':>9} {'Max cap':>9} {'Values clamped':>14}")
print("-" * 78)

for col in cap_cols_present:
    lo, hi = PHYSIOLOGICAL_CAPS[col]
    before = df[col].copy()
    df[col] = df[col].clip(lower=lo, upper=hi)
    n_clamped = (before != df[col]).sum()
    total_capped += n_clamped
    lo_str = str(lo) if lo is not None else "—"
    hi_str = str(hi) if hi is not None else "—"
    print(f"{col:<42} {lo_str:>9} {hi_str:>9} {n_clamped:>14}")

print(f"\nTotal values clamped: {total_capped}")

# --- Data quality summary ---
print("\nData quality summary after cleaning (numeric columns, excluding target):")
numeric_df = df.select_dtypes(include="number").drop(columns=["stroke"], errors="ignore")
summary = pd.DataFrame({
    "missing": numeric_df.isna().sum(),
    "min":     numeric_df.min(),
    "max":     numeric_df.max(),
    "median":  numeric_df.median(),
}).sort_values("missing", ascending=False)
print(summary.to_string())

Physiological hard caps applied:
Column                                       Min cap   Max cap Values clamped
------------------------------------------------------------------------------
energy                                           400      6000             33
protein                                            5       280             15
Carbohydrate                                       5       700             26
Dietary fiber                                      1        70             23
Total saturated fatty acids                      0.5       100             17
Total monounsaturated fatty acids                0.5       100             22
Total polyunsaturated fatty acids                0.5        75             21
Potassium                                        200      8000             19
Sodium                                           300     12000             18
Glycohemoglobin                                  3.5      18.0              1

Total values clamped: 195

Da

## 3c. Feature engineering

Two features constructed from existing columns before the split.
All are arithmetic combinations so there is no leakage.

**Multimorbidity** = sum of (diabetes + hypertension + high cholesterol + smoke)
- Simple count of how many concurrent risk conditions are present (0–4)
- Stroke rate rises monotonically with count: 1.1% (0) → 4.7% (1) → 7.2% (2) → 10.0% (3) → 13.8% (4)
- Captures cumulative risk burden that individual binary flags miss; also helps regularization
  concentrate weight on the aggregate rather than splitting it across 4 correlated binary terms

**Age interaction terms** (age × hypertension, age × smoke, age × high cholesterol)
- Logistic regression is a linear model — it cannot learn interactions without explicit features
- Age group amplifies the effect of each risk condition: being older with hypertension carries more stroke risk than either factor alone
- Correlations with stroke: age×hyp=0.11, age×smoke=0.10, age×high_chol=0.09

Note: pulse pressure (Systolic − Diastolic) was tested but dropped. It is a linear combination
of two features already in the model, so a linear model can already extract that signal through
the raw Systolic and Diastolic coefficients. Adding it creates pure collinearity without new information.

In [26]:
# --- Multimorbidity count ---
# Cumulative risk burden — stroke rate rises monotonically 1.1% → 13.8% as count goes 0→4
# Also helps regularization concentrate weight on the aggregate rather than splitting it
# across 4 individual correlated binary terms
_risk_flags = ["diabetes", "hypertension", "high cholesterol", "smoke"]
_present = [c for c in _risk_flags if c in df.columns]
df["multimorbidity"] = df[_present].sum(axis=1)

# --- Age interaction terms ---
# Logistic regression cannot learn interactions without explicit construction
df["age_x_hypertension"]     = df["age"] * df["hypertension"]
df["age_x_smoke"]            = df["age"] * df["smoke"]
df["age_x_high_cholesterol"] = df["age"] * df["high cholesterol"]

engineered = ["multimorbidity", "age_x_hypertension", "age_x_smoke", "age_x_high_cholesterol"]

print("Engineered features added:")
for feat in engineered:
    corr = df[feat].corr(df["stroke"])
    print(f"  {feat:<30}  corr={corr:.4f}")
print(f"\nDataset shape after feature engineering: {df.shape}")

Engineered features added:
  multimorbidity                  corr=0.1077
  age_x_hypertension              corr=0.1100
  age_x_smoke                     corr=0.0987
  age_x_high_cholesterol          corr=0.0923

Dataset shape after feature engineering: (4603, 40)


## 4. Define label and features

- `y` is the label we want to predict: `stroke`
- `X` is everything else

We also drop leakage columns here before splitting so they never reach the model.

We print label balance because stroke is rare. That is why accuracy can be misleading.


In [27]:
if "stroke" not in df.columns:
    raise ValueError("Dataset must contain a 'stroke' column.")

# Drop leakage columns if present
drop_cols = [c for c in LEAKAGE_COLS if c in df.columns]
df_clean = df.drop(columns=drop_cols)

# Drop columns excluded for UX/practical reasons (require lab tests)
exclude_present = [c for c in EXCLUDE_COLS if c in df_clean.columns]
df_clean = df_clean.drop(columns=exclude_present)

# Drop redundant columns (near-perfect multicollinearity with other features)
redundant_present = [c for c in REDUNDANT_COLS if c in df_clean.columns]
df_clean = df_clean.drop(columns=redundant_present)

y = df_clean["stroke"]
X = df_clean.drop(columns=["stroke"])

print("Dropped leakage columns:   ", drop_cols)
print("Dropped excluded columns:  ", exclude_present)
print("Dropped redundant columns: ", redundant_present)
print(f"\nFeatures remaining: {X.shape[1]}")
print()
print("Label counts")
print(y.value_counts(dropna=False))
print()
print("Label proportions")
print(y.value_counts(normalize=True, dropna=False))

Dropped leakage columns:    ['General health condition', 'depression', 'Minutes sedentary activity', 'Coronary Heart Disease']
Dropped excluded columns:   ['High-density lipoprotein', 'Triglyceride', 'Low-density lipoprotein']
Dropped redundant columns:  ['Total fat']

Features remaining: 31

Label counts
stroke
0    4241
1     362
Name: count, dtype: int64

Label proportions
stroke
0    0.921356
1    0.078644
Name: proportion, dtype: float64


## 5. Cross-validation setup

We evaluate with fixed 5-fold stratified cross-validation.

- Each fold keeps stroke prevalence close to the full dataset.
- Each row receives one out-of-fold prediction from a model that never trained on that row.
- We aggregate out-of-fold predictions for the final CV metrics.


In [28]:
cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=RANDOM_STATE,
)

print(
    f"Evaluation setup: {CV_FOLDS}-fold stratified CV "
    f"(shuffle={CV_SHUFFLE}, random_state={RANDOM_STATE})"
)
print("Fold label balance:")
for fold_idx, (_, valid_idx) in enumerate(cv.split(X, y), start=1):
    y_valid_fold = y.iloc[valid_idx]
    positives = int(y_valid_fold.sum())
    prevalence = float(y_valid_fold.mean())
    print(f"  Fold {fold_idx}: n={len(valid_idx)} | positives={positives} | prevalence={prevalence:.4f}")


Evaluation setup: 5-fold stratified CV (shuffle=True, random_state=42)
Fold label balance:
  Fold 1: n=921 | positives=72 | prevalence=0.0782
  Fold 2: n=921 | positives=73 | prevalence=0.0793
  Fold 3: n=921 | positives=73 | prevalence=0.0793
  Fold 4: n=920 | positives=72 | prevalence=0.0783
  Fold 5: n=920 | positives=72 | prevalence=0.0783


## 5b. Outlier clipping

Extreme values in numeric columns (e.g. Sodium=7, very high triglycerides) can distort StandardScaler and destabilize logistic regression.

We use IQR-based clipping inside each CV fold:
- Bounds are computed on the **training fold only** and then applied to train/validation rows in that fold
- `lower = Q1 - k * IQR`, `upper = Q3 + k * IQR`
- `k = CLIP_IQR_MULTIPLIER` (default 2.0)

Categorical columns are not clipped.


In [29]:
# Prepare numeric columns for optional IQR clipping (fit on train fold only)
all_cat_cols = ORDINAL_COLS + NOMINAL_CATEGORICAL_COLS
clip_num_cols = [c for c in X.columns if c not in all_cat_cols]


def apply_iqr_clipping(train_df, valid_df, numeric_cols, k):
    if not numeric_cols:
        return train_df.copy(), valid_df.copy(), 0

    q1 = train_df[numeric_cols].quantile(0.25)
    q3 = train_df[numeric_cols].quantile(0.75)
    iqr = q3 - q1
    lower_bounds = q1 - k * iqr
    upper_bounds = q3 + k * iqr

    train_out = train_df.copy()
    valid_out = valid_df.copy()

    train_out[numeric_cols] = train_out[numeric_cols].clip(
        lower=lower_bounds, upper=upper_bounds, axis=1
    )
    valid_out[numeric_cols] = valid_out[numeric_cols].clip(
        lower=lower_bounds, upper=upper_bounds, axis=1
    )

    clipped_low = (train_df[numeric_cols] < lower_bounds).sum()
    clipped_high = (train_df[numeric_cols] > upper_bounds).sum()
    total_clipped = int((clipped_low + clipped_high).sum())

    return train_out, valid_out, total_clipped


if CLIP_OUTLIERS:
    print(
        f"Outlier clipping enabled inside CV folds "
        f"(k={CLIP_IQR_MULTIPLIER}) on {len(clip_num_cols)} numeric columns."
    )
else:
    print("Outlier clipping skipped (CLIP_OUTLIERS=False in config).")


Outlier clipping enabled inside CV folds (k=2.0) on 24 numeric columns.


## 6. Preprocessing

Three streams based on what we learned from the data:

**Ordinal stream** (`age`): impute most frequent → OrdinalEncoder
- Age has a strong monotonic relationship with stroke (1.9% → 6.1% → 12.3%)
- OrdinalEncoder preserves this rank order; one-hot would lose it

**Nominal categorical stream** (gender, Race, Marital status, sleep disorder, Health Insurance, BMI): impute most frequent → OneHotEncoder
- BMI stays here because its stroke rate is non-monotonic (confounded by age) so one-hot is more appropriate than ordinal

**Numeric stream** (all remaining columns including binary 0/1 flags): impute median → StandardScaler
- Binary columns (alcohol, smoke, diabetes, hypertension, high cholesterol, Coronary Heart Disease) are already 0/1 — no encoding needed, and one-hot would just create perfectly collinear dummy pairs

In [30]:
# --- Assign columns to streams ---
ord_cols  = [c for c in ORDINAL_COLS             if c in X.columns]
nom_cols  = [c for c in NOMINAL_CATEGORICAL_COLS if c in X.columns]
# Everything else goes to numeric (includes binary 0/1 columns)
num_cols  = [c for c in X.columns if c not in ord_cols + nom_cols]

print("Ordinal columns:          ", ord_cols)
print("Nominal categorical cols: ", nom_cols)
print("Numeric columns:          ", num_cols)
print(f"\nTotal: {len(ord_cols)} ordinal + {len(nom_cols)} nominal + {len(num_cols)} numeric = {len(X.columns)} features")

# --- Pipelines ---
ord_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=ORDINAL_CATEGORIES)),
])

nom_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

preprocess = ColumnTransformer(
    transformers=[
        ("ord", ord_pipe, ord_cols),
        ("nom", nom_pipe, nom_cols),
        ("num", num_pipe, num_cols),
    ]
)

Ordinal columns:           ['age']
Nominal categorical cols:  ['gender', 'Race', 'Marital status', 'sleep disorder', 'Health Insurance', 'Body Mass Index']
Numeric columns:           ['alcohol', 'smoke', 'sleep time', 'diabetes', 'hypertension', 'high cholesterol', 'Waist Circumference', 'Systolic blood pressure', 'Diastolic blood pressure', 'Fasting Glucose', 'Glycohemoglobin', 'energy', 'protein', 'Carbohydrate', 'Dietary fiber', 'Total saturated fatty acids', 'Total monounsaturated fatty acids', 'Total polyunsaturated fatty acids', 'Potassium', 'Sodium', 'multimorbidity', 'age_x_hypertension', 'age_x_smoke', 'age_x_high_cholesterol']

Total: 1 ordinal + 6 nominal + 24 numeric = 31 features


## 7. Model

We use **logistic regression with elastic net regularization**.

Elastic net combines two penalty types:
- **L1 (lasso)**: pushes less useful feature weights to exactly zero — built-in feature selection
- **L2 (ridge)**: shrinks correlated features together instead of picking one arbitrarily

With our data this is useful because:
- We have many correlated dietary features (e.g. energy, protein, carbohydrate, fat all move together)
- Some features may carry very little signal after leakage columns are removed
- L1 zeroes those out; L2 handles the correlated group structure

`l1_ratio=0.5` gives an equal mix. `solver="saga"` is required for elastic net in sklearn.

We keep `class_weight="balanced"` because stroke is rare (~5% of data).
The model outputs probabilities with `predict_proba` — those are the risk scores.

In [31]:
model = LogisticRegression(
    penalty=PENALTY,
    C=C,
    l1_ratio=L1_RATIO,
    solver=SOLVER,
    max_iter=MAX_ITER,
    class_weight=CLASS_WEIGHT,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", model),
    ]
)

pipeline

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('ord',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ordinal',
                                                                   OrdinalEncoder(categories=[[1,
                                                                                               2,
                                                                                               3]]))]),
                                                  ['age']),
                                                 ('nom',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gend...
                                                   'Dietary fiber',
                                                   'Total saturated fatty '
                                                   'acids',
                                                   'Total monounsaturated '
                                                   'fatty acids',
                                                   'Total polyunsaturated '
                                                   'fatty acids',
                                                   'Potassium', 'Sodium',
                                                   'multimorbidity',
                                                   'age_x_hypertension',
                                                   'age_x_smoke',
                                                   'age_x_high_cholesterol'])])),
                ('model',
                 LogisticRegression(C=1, class_weight='balanced', l1_ratio=0.5,
                                    max_iter=2000, penalty='elasticnet',
                                    random_state=42, solver='saga'))])

## 8. Train

This fits the preprocessing and the model on the training set only.


In [32]:
# Out-of-fold risk scores across all 5 folds
y_prob = np.zeros(len(y), dtype=float)
fold_metrics = []

for fold_idx, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx].copy()
    X_valid_fold = X.iloc[valid_idx].copy()
    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    if CLIP_OUTLIERS:
        X_train_fold, X_valid_fold, clipped_values = apply_iqr_clipping(
            X_train_fold,
            X_valid_fold,
            clip_num_cols,
            CLIP_IQR_MULTIPLIER,
        )
    else:
        clipped_values = 0

    fold_pipeline = clone(pipeline)
    fold_pipeline.fit(X_train_fold, y_train_fold)

    fold_prob = fold_pipeline.predict_proba(X_valid_fold)[:, 1]
    y_prob[valid_idx] = fold_prob

    fold_pred = (fold_prob >= THRESHOLD).astype(int)
    fold_accuracy = float(accuracy_score(y_valid_fold, fold_pred))
    fold_precision = float(precision_score(y_valid_fold, fold_pred, zero_division=0))
    fold_recall = float(recall_score(y_valid_fold, fold_pred, zero_division=0))
    fold_auc = float(roc_auc_score(y_valid_fold, fold_prob))

    fold_metrics.append(
        {
            "fold": fold_idx,
            "n_valid": int(len(valid_idx)),
            "positives": int(y_valid_fold.sum()),
            "accuracy": fold_accuracy,
            "precision": fold_precision,
            "recall": fold_recall,
            "auc": fold_auc,
            "clipped_values": int(clipped_values),
        }
    )

    print(
        f"Fold {fold_idx}/{CV_FOLDS} | n={len(valid_idx)} | pos={int(y_valid_fold.sum())} "
        f"| acc={fold_accuracy:.4f} | prec={fold_precision:.4f} "
        f"| rec={fold_recall:.4f} | auc={fold_auc:.4f} "
        f"| clipped={int(clipped_values)}"
    )

# Fit once on full data for coefficients + exported pipeline artifact
if CLIP_OUTLIERS:
    X_for_fit, _, clipped_full = apply_iqr_clipping(
        X,
        X,
        clip_num_cols,
        CLIP_IQR_MULTIPLIER,
    )
    print(f"Final fit clipping on full dataset: {clipped_full} values clipped.")
else:
    X_for_fit = X.copy()

pipeline.fit(X_for_fit, y)
print("Final pipeline fitted on full dataset.")


/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Fold 1/5 | n=921 | pos=72 | acc=0.3616 | prec=0.1067 | rec=0.9722 | auc=0.6975 | clipped=1872


/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Fold 2/5 | n=921 | pos=73 | acc=0.3626 | prec=0.1046 | rec=0.9315 | auc=0.7022 | clipped=1859


/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Fold 3/5 | n=921 | pos=73 | acc=0.3464 | prec=0.0998 | rec=0.9041 | auc=0.6442 | clipped=1883


/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Fold 4/5 | n=920 | pos=72 | acc=0.3391 | prec=0.0939 | rec=0.8611 | auc=0.6373 | clipped=1912


/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Fold 5/5 | n=920 | pos=72 | acc=0.3315 | prec=0.0917 | rec=0.8472 | auc=0.6978 | clipped=1895
Final fit clipping on full dataset: 2335 values clipped.
Final pipeline fitted on full dataset.


## 9. Evaluate with out-of-fold CV predictions

Important definitions:
- `y_prob` is the out-of-fold predicted probability of stroke. This is the risk score.
- `THRESHOLD` turns probability into a yes or no prediction for metrics.

Metrics in simple language:
- Accuracy: percent correct overall
- Precision: when we flag stroke, how often we are right
- Recall: out of true stroke cases, how many we catch
- ROC AUC: how well the risk scores rank stroke above non stroke across all possible thresholds

Confusion matrix format:
[[TN FP]
 [FN TP]]

For a prevention demo, recall matters a lot. Missing stroke cases is worse than extra false alarms in an educational demo.


In [33]:
# Turn out-of-fold risk scores into predictions using the chosen threshold
y_pred = (y_prob >= THRESHOLD).astype(int)

accuracy = float(accuracy_score(y, y_pred))
precision = float(precision_score(y, y_pred, zero_division=0))
recall = float(recall_score(y, y_pred, zero_division=0))
auc = float(roc_auc_score(y, y_prob))
cm = confusion_matrix(y, y_pred)

print(f"Threshold {THRESHOLD}")
print(f"CV method: {CV_FOLDS}-fold stratified (out-of-fold)")
print("Accuracy", accuracy)
print("Precision", precision)
print("Recall", recall)
print("ROC AUC", auc)
print("Confusion matrix")
print(cm)


Threshold 0.3
CV method: 5-fold stratified (out-of-fold)
Accuracy 0.34825114056050405
Precision 0.09933171324422843
Recall 0.9033149171270718
ROC AUC 0.6759116803735177
Confusion matrix
[[1276 2965]
 [  35  327]]


## 9b. Feature weights (elastic net coefficients)

Elastic net with L1 pushes low-signal features to exactly zero.
This shows which features survived regularization and their signed contribution to stroke risk.

In [34]:
# Retrieve feature names after preprocessing
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
coefs = pipeline.named_steps["model"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
)

zeroed = (coef_df["coefficient"] == 0).sum()
print(f"Features zeroed by L1: {zeroed} / {len(coef_df)}")
print()
print("Top 15 by absolute coefficient:")
print(coef_df.head(15).to_string(index=False))
print()
print("Zeroed-out features:")
print(coef_df[coef_df["coefficient"] == 0]["feature"].tolist())

Features zeroed by L1: 2 / 46

Top 15 by absolute coefficient:
                         feature  coefficient  abs_coef
             num__multimorbidity     1.097344  1.097344
                        ord__age     1.070841  1.070841
                     num__energy    -0.733705  0.733705
               num__Carbohydrate     0.529016  0.529016
           nom__Marital status_4     0.353193  0.353193
                     nom__Race_2    -0.343723  0.343723
num__Total saturated fatty acids     0.326092  0.326092
                   num__diabetes    -0.300158  0.300158
                     nom__Race_4     0.298572  0.298572
           nom__Marital status_1    -0.271094  0.271094
           nom__Marital status_3    -0.255709  0.255709
           num__high cholesterol    -0.250174  0.250174
         num__age_x_hypertension    -0.248448  0.248448
            num__Glycohemoglobin    -0.231122  0.231122
          nom__Body Mass Index_3    -0.211311  0.211311

Zeroed-out features:
['nom__Race_1', 'nu

## 10. Score distribution cutoffs (for risk labels)

Docs and Content uses the model score distribution to define low, medium, and high risk bins.
We report:
- min and max
- median (50th percentile)
- 80th percentile
- 95th percentile

Percentile meaning:
- 80th percentile means the score is higher than about 80 percent of people in the CV out-of-fold score distribution.

These cutoffs are for demo labeling only. They are not clinical thresholds.


In [35]:
cutoffs = {
    "min": float(np.min(y_prob)),
    "median_p50": float(np.percentile(y_prob, 50)),
    "p80": float(np.percentile(y_prob, 80)),
    "p95": float(np.percentile(y_prob, 95)),
    "max": float(np.max(y_prob)),
}

for k, v in cutoffs.items():
    print(f"{k}: {v:.6f}")


min: 0.016345
median_p50: 0.437170
p80: 0.625333
p95: 0.751644
max: 0.910096


## 11. Quick threshold scan (precision and recall tradeoff)

This prints a few thresholds so we can choose a good demo default.
Lower thresholds increase recall and decrease precision.

This does not change ROC AUC because AUC uses the full probability ranking.


In [36]:
threshold_scan_results = []

for t in THRESHOLD_SCAN:
    yp = (y_prob >= t).astype(int)
    p = float(precision_score(y, yp, zero_division=0))
    r = float(recall_score(y, yp, zero_division=0))
    c = confusion_matrix(y, yp)

    threshold_scan_results.append(
        {
            "threshold": float(t),
            "precision": p,
            "recall": r,
            "confusion_matrix": c.tolist(),
        }
    )

    print(f"threshold={t:.2f} | precision={p:.4f} | recall={r:.4f} | cm={c.tolist()}")


threshold=0.05 | precision=0.0792 | recall=0.9972 | cm=[[45, 4196], [1, 361]]
threshold=0.10 | precision=0.0811 | recall=0.9945 | cm=[[161, 4080], [2, 360]]
threshold=0.20 | precision=0.0892 | recall=0.9779 | cm=[[627, 3614], [8, 354]]
threshold=0.30 | precision=0.0993 | recall=0.9033 | cm=[[1276, 2965], [35, 327]]


## 12. Save report

This writes a single markdown report to keep results easy to find.
It includes the CV settings, leakage columns dropped, threshold used, metrics, confusion matrix, per-fold metrics, and score cutoffs.

File:
- `reports/baseline_metrics.md`


In [37]:
REPORT_PATH = ROOT / REPORT_REL_PATH
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# Baseline Metrics (Final Demo Baseline)\n\n")

    f.write("## Model\n")
    f.write(
        f"Logistic Regression with Elastic Net (penalty={PENALTY}, C={C}, "
        f"l1_ratio={L1_RATIO}, solver={SOLVER})\n\n"
    )

    f.write("## Evaluation\n")
    f.write("method: stratified_k_fold_cv\n")
    f.write(f"folds: {CV_FOLDS}\n")
    f.write(f"shuffle: {'yes' if CV_SHUFFLE else 'no'}\n")
    f.write(f"random_state: {RANDOM_STATE}\n")
    f.write("stratify: yes\n")
    f.write(f"outlier_clipping: {'yes' if CLIP_OUTLIERS else 'no'}\n")
    if CLIP_OUTLIERS:
        f.write(f"clip_iqr_multiplier: {CLIP_IQR_MULTIPLIER}\n")
    f.write("\n")

    f.write("## Leakage handling\n")
    f.write("Dropped columns\n")
    for c in drop_cols:
        f.write(f"- {c}\n")
    f.write("\n")

    f.write("## Threshold\n")
    f.write(f"threshold: {THRESHOLD}\n\n")

    f.write("## Label balance\n")
    f.write("Counts\n")
    f.write(y.value_counts(dropna=False).to_string())
    f.write("\n\n")

    f.write("## Metrics from 5-fold stratified CV (out-of-fold)\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")
    f.write(f"ROC AUC: {auc:.4f}\n\n")

    f.write("## Confusion matrix from 5-fold stratified CV (out-of-fold)\n")
    f.write("Format is [[TN FP]\n")
    f.write("           [FN TP]]\n\n")
    f.write(np.array2string(cm))
    f.write("\n\n")

    f.write("## Per-fold metrics\n")
    for row in fold_metrics:
        f.write(
            f"- fold={row['fold']} | n={row['n_valid']} | positives={row['positives']} | "
            f"accuracy={row['accuracy']:.4f} | precision={row['precision']:.4f} | "
            f"recall={row['recall']:.4f} | auc={row['auc']:.4f} | "
            f"clipped_values={row['clipped_values']}\n"
        )
    f.write("\n")

    f.write("## Score cutoffs from CV out-of-fold probabilities\n")
    for k, v in cutoffs.items():
        f.write(f"- {k}: {v:.6f}\n")
    f.write("\n")

    f.write("## Threshold scan (CV out-of-fold)\n")
    for row in threshold_scan_results:
        f.write(
            f"- threshold={row['threshold']:.2f} | "
            f"precision={row['precision']:.4f} | "
            f"recall={row['recall']:.4f} | "
            f"cm={row['confusion_matrix']}\n"
        )

print(f"Saved report to {REPORT_PATH}")


Saved report to /Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/reports/baseline_metrics.md


## Export the trained pipeline for the app

If you want the Streamlit app to load a trained model without retraining, save the pipeline with joblib.

This creates:
- `models/baseline_pipeline.joblib`

Before running, make sure repo plan allows committing or storing model artifacts.


In [38]:
import joblib

models_dir = ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "baseline_pipeline.joblib"
joblib.dump(pipeline, model_path)
print(f"Saved pipeline to {model_path}")

Saved pipeline to /Users/darylokeke/Desktop/has_projects/stroke-prevention-demo/models/baseline_pipeline.joblib
